# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

load_dotenv(override=True)
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
MODEL = 'llama3.2'

openai = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [2]:
# Initialize and constants

# load_dotenv(override=True)
# api_key = os.getenv('OPENAI_API_KEY')

# if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
#     print("API key looks good so far")
# else:
#     print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = 'gpt-5-nano'
# openai = OpenAI()

In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/05/28/connecting-my-courses-become-an-llm-expert-and-leader/',
 'https://edwarddonner.com/2025/05/28/connecting-my-cou

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/11/11/ai-live-event/
https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/
htt

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home link', 'url': 'https://edwarddonner.com/'},
  {'type': 'connect four page',
   'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'outsmart page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'about me and about nebula page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home link', 'url': 'https://edwarddonner.com/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling llama3.2
Found 4 relevant links


{'links': [{'type': 'about company', 'url': 'https://edwarddonner.com'},
  {'type': 'personal blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 15 relevant links


{'links': [{'type': 'company website', 'url': 'https://huggingface.co'},
  {'type': 'Products page', 'url': '/models'},
  {'type': 'Datasets page', 'url': '/datasets'},
  {'type': 'Spaces page', 'url': '/spaces'},
  {'type': 'Docs page', 'url': '/docs'},
  {'type': 'Enterprise products', 'url': '/enterprise'},
  {'type': 'Pricing page', 'url': '/pricing'},
  {'type': 'Company pages', 'url': 'https://allenai.org/'},
  {'type': 'Company pages', 'url': 'https://facebook.com/'},
  {'type': 'Company pages', 'url': 'https://amazon.com/'},
  {'type': 'Company pages', 'url': 'https://google.com/'},
  {'type': 'Company pages', 'url': 'https://intel.com/'},
  {'type': 'Company pages', 'url': 'https://microsoft.com/'},
  {'type': 'Company pages', 'url': 'https://grammarly.com/'},
  {'type': 'Company pages', 'url': 'https://writer.com/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-4.7
Updated
6 days ago
•
28k
•
1.13k
MiniMaxAI/MiniMax-M2.1
Updated
1 day ago
•
45.3k
•
503
Qwen/Qwen-Image-Edit-2511
Updated
5 days ago
•
16.6k
•
488
Qwen/Qwen-Image-Layered
Updated
10 days ago
•
15.3k
•
812
google/functiongemma-270m-it
Updated
10 days ago
•
35.4k
•
672
Browse 2M+ models
Spaces
Running
Featured
3.12k
Wan2.2 Animate
👁
3.12k
Wan2.2 Animate
Running
on
Zero
Featured
616
TRELLIS.2
🏢
616
High-fidelity 3D Generation from images
Running
on
Zero
Featured
309
Qwen Image Layered
🚀
309
Decompose an image into layers an

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("Xensam", "https://xensam.com")

Selecting relevant links for https://xensam.com by calling llama3.2
Found 5 relevant links


'\nYou are looking at a company called: Xensam\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\n403 - Forbidden\n\nbg_error_lines\ncircle_dots\n403 - Forbidden\nAccess to this page is forbidden.\nclouds_shape\n## Relevant Links:\n\n\n### Link: about page\n403 - Forbidden\n\nbg_error_lines\ncircle_dots\n403 - Forbidden\nAccess to this page is forbidden.\nclouds_shape\n\n### Link: home page\n403 - Forbidden\n\nbg_error_lines\ncircle_dots\n403 - Forbidden\nAccess to this page is forbidden.\nclouds_shape\n\n### Link: careers page\n403 - Forbidden\n\nbg_error_lines\ncircle_dots\n403 - Forbidden\nAccess to this page is forbidden.\nclouds_shape\n\n### Link: contact us page\n403 - Forbidden\n\nbg_error_lines\ncircle_dots\n403 - Forbidden\nAccess to this page is forbidden.\nclouds_shape\n\n### Link: investors page\n403 - Forbidden\n\nbg_error_lines\ncircle

In [20]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model= MODEL,
        messages= [
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("Xensam", "https://xensam.com")

Selecting relevant links for https://xensam.com by calling llama3.2
Found 5 relevant links


It appears that the Xensam website is not yet fully functional or open due to access restrictions. I'll create a brochure based on what can be inferred about the company from its absence of any official information.

**Xensam**

A pioneering company driven by innovation and collaboration.

*Founded in [year]*, Xensam has been working tirelessly behind the scenes to develop cutting-edge solutions for [industry/market].
Their dedication to excellence and commitment to making a lasting impact is evident throughout their approach.
Although the website is currently unavailable due to security restrictions, it's clear that Xensam prioritizes protecting sensitive information while being open about their mission.

**Our Values**

Trust: We value honesty and transparency in all our interactions.
Innovation: We believe embracing new ideas leads to transformative breakthroughs.
Collaboration: Unity among team members fosters a creative environment.

*Although no official company culture is evident, it's likely that Xensam encourages employees to be open-minded, engaged, and respectful of differing opinions.*
A supportive community with opportunities for professional growth would make sense given the industry and job title hints provided on their careers web page:
> IT/Software Development Jobs
The hiring team at Xensam seems to focus on finding individuals skilled in a range of fields such as programming languages (e.g., Java), data analysis, and cloud architecture.
We look forward to unlocking the potential that lies within our team members.

A company committed to driving change and fostering growth will make a lasting impression when the time is right.
For now, we invite curious minds to stay tuned for updates regarding their vision and endeavors.

Follow Xensam on social media with an online presence waiting to be discovered.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
stream_brochure("Xensam", "https://xensam.com")

Selecting relevant links for https://xensam.com by calling llama3.2
Found 3 relevant links


It seems that the Xensam website is intentionally not providing much information about itself. However, I'll try to create a brief brochure based on what's generally known about companies and provide some general information about company culture.

**Xensam: A Leader in [Industry/Field]**

Although Xensam's official website doesn't reveal much about the company, it can be inferred that they operate in a competitive field. Xensam is dedicated to innovation and strives for excellence in their industry.

**Company Culture at Xensam**

Xensam values collaboration and integrity. A dynamic work environment where team members are encouraged to think creatively and share ideas will be key factors in driving success. The company fosters open communication, promoting transparency and trust with all employees.

**Serving Valued Customers**

Although we cannot confirm specific lines of business or industries for Xensam, the company aims to serve its clients by providing a wide range of valuable services that are designed to meet their diverse requirements. Our top priority is meeting customer needs while maintaining reliability.

**Join Our Team at Xensam**

Considering the absence of information regarding job openings on the official website, but recognizing the importance of talent for driving forward innovation and progress, Xensam may be currently seeking employees with relevant skills to take part in their mission to grow.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>